In [ ]:
# Cell 1 [挂载 Google Drive 与 GPU 体检]
import os
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/DCPA_Project")
for sub in ("code", "data_archive", "checkpoints", "logs", "export"):
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
print(f"[Drive] Ready: {DRIVE_ROOT}")

# GPU health check & 4-tier hardware sharding preview (no hardcoded torch.cuda.get_device_properties)
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True,
    text=True,
    check=False,
)
if smi.returncode == 0 and smi.stdout.strip():
    first_line = smi.stdout.strip().splitlines()[0]
    gpu_name, mem_mib_s = [part.strip() for part in first_line.split(",", 1)]
    vram_gb = float(mem_mib_s) / 1024.0
    if vram_gb >= 75.0:
        tier, micro_bytes, accum = "h100_or_a100_80g", 131072, 1
    elif vram_gb >= 38.0:
        tier, micro_bytes, accum = "a100_40g", 65536, 2
    elif vram_gb >= 20.0:
        tier, micro_bytes, accum = "l4_or_4090", 32768, 4
    else:
        tier, micro_bytes, accum = "cuda_compact", 16384, 8
    print(
        f"[GPU Health] {gpu_name} | VRAM={vram_gb:.1f} GB | tier={tier} | "
        f"micro_batch_bytes={micro_bytes} | grad_accum_steps={accum} | precision=bfloat16"
    )
else:
    print("[GPU Health][WARN] No NVIDIA GPU detected via nvidia-smi; will fall back to CPU float32.")


In [ ]:
# Cell 2 [部署代码至本地 NVMe 与安装依赖]
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

BUNDLE_ZIP = Path("/content/drive/MyDrive/DCPA_Project/code/dcpa_colab_bundle.zip")
CODE_DIR = Path("/content/dcpa")
NVME_CACHE = Path("/content/nvme_cache")

if not BUNDLE_ZIP.exists():
    raise FileNotFoundError(
        f"Missing code bundle at {BUNDLE_ZIP}. Run `bash scripts/pack_for_colab.sh` locally "
        "and upload `dcpa_colab_bundle.zip` to Google Drive first."
    )

if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
CODE_DIR.mkdir(parents=True, exist_ok=True)
NVME_CACHE.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(BUNDLE_ZIP, "r") as zf:
    zf.extractall(CODE_DIR)

os.chdir(CODE_DIR)
for p in (str(CODE_DIR), str(CODE_DIR / "dcpa-train-py")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ["PYTHONPATH"] = f"{CODE_DIR}:{CODE_DIR / 'dcpa-train-py'}"

deps = [
    "safetensors",
    "datasets",
    "hf_transfer",
    "tree-sitter",
    "tree-sitter-python",
    "tree-sitter-rust",
    "pyyaml",
    "pytest",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *deps], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--no-build-isolation", "-e", str(CODE_DIR / "dcpa-train-py")],
    check=True,
)

from utils.device import describe_hardware_profile, detect_hardware_profile

profile = detect_hardware_profile("auto")
print(f"[Deploy] Code unpacked to {CODE_DIR} | HF_HUB_ENABLE_HF_TRANSFER={os.environ['HF_HUB_ENABLE_HF_TRANSFER']}")
print(describe_hardware_profile(profile))


In [ ]:
# Cell 3 [语料智能缓存：首次流式构建归档，重连 3 秒极速恢复]
import os
import shutil
import subprocess
import sys
import tarfile
import time
from pathlib import Path

from train.colab_drive_sync import sync_file_to_drive

DRIVE_ARCHIVE = Path("/content/drive/MyDrive/DCPA_Project/data_archive/dcpa_corpus_v1.tar.gz")
NVME_DATA_DIR = Path("/content/nvme_cache/data")
NVME_TAR_TMP = Path("/content/nvme_cache/dcpa_corpus_v1.tar.gz")

NVME_DATA_DIR.mkdir(parents=True, exist_ok=True)
t0 = time.perf_counter()

if DRIVE_ARCHIVE.exists():
    print(f"[Corpus Cache HIT] Restoring {DRIVE_ARCHIVE} -> {NVME_DATA_DIR} ...")
    shutil.copy2(DRIVE_ARCHIVE, NVME_TAR_TMP)
    with tarfile.open(NVME_TAR_TMP, "r:gz") as tf:
        tf.extractall(NVME_DATA_DIR)
    NVME_TAR_TMP.unlink(missing_ok=True)
    print(f"[Corpus Cache HIT] Restored in {time.perf_counter() - t0:.2f}s (skipped re-download).")
else:
    print("[Corpus Cache MISS] Building full ~450 MB 5-domain corpus directly on local NVMe...")
    subprocess.run(
        [sys.executable, "data/build_corpus.py", "--out-dir", str(NVME_DATA_DIR)],
        check=True,
    )
    env = dict(os.environ)
    env["DCPA_DATA_DIR"] = str(NVME_DATA_DIR)
    subprocess.run(
        [sys.executable, "-m", "pytest", "-v", "tests/test_data_pipeline.py"],
        env=env,
        check=True,
    )
    print("[Corpus] All 5 data pipeline assertions PASSED. Archiving dcpa_corpus_v1.tar.gz to Drive...")
    with tarfile.open(NVME_TAR_TMP, "w:gz") as tf:
        for fname in ("train.bin", "train.idx", "train.meta.json", "val.bin", "val.idx", "val.meta.json", "corpus_meta.json"):
            fpath = NVME_DATA_DIR / fname
            if fpath.exists():
                tf.add(fpath, arcname=fname)
    sync_file_to_drive(NVME_TAR_TMP, DRIVE_ARCHIVE.parent, dest_name=DRIVE_ARCHIVE.name)
    NVME_TAR_TMP.unlink(missing_ok=True)
    if hasattr(os, "sync"):
        os.sync()
    print(f"[Corpus] Archived to {DRIVE_ARCHIVE} in {time.perf_counter() - t0:.1f}s.")


In [ ]:
# Cell 4 [云端 CUDA 算子 20 秒上线复检]
import subprocess
import sys
import time

t0 = time.perf_counter()
subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-v",
        "tests/test_sanity.py",
        "tests/test_trainer_lifecycle.py",
    ],
    check=True,
)
print(f"[Cloud Preflight] Causality, STE gradients & A/B Drive sync 100% GREEN in {time.perf_counter() - t0:.1f}s.")


In [ ]:
# Cell 5 [一键启动 / 断连无缝续跑 10 万步 + 完工自动释放显卡省算力]
# ==============================================================================
# AUTO_UNASSIGN_RUNTIME_ON_FINISH:
#   True  -> 训练跑满 100,000 步导出 dcpa_final.safetensors 至 Drive（或触发 3 次
#            不可恢复熔断停机）后，强制 os.sync() 等待 10 秒并自动归还 H100/A100
#            实例，防止半夜跑完后空烧 Colab 计算单元（CCU）！
#   False -> 训练结束后保留当前 Colab 实例以便手动调试。
# ==============================================================================
AUTO_UNASSIGN_RUNTIME_ON_FINISH = True

import os
import subprocess
import sys
import time
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/DCPA_Project")
NVME_CACHE = Path("/content/nvme_cache")
NVME_DATA_DIR = NVME_CACHE / "data"
DRIVE_CKPTS = DRIVE_ROOT / "checkpoints"
DRIVE_LOGS = DRIVE_ROOT / "logs"
DRIVE_EXPORT = DRIVE_ROOT / "export"
FINAL_SAFETENSORS = DRIVE_EXPORT / "dcpa_final.safetensors"

cmd = [
    sys.executable,
    "-u",
    "train/train_dcpa.py",
    "--config", "configs/dcpa_48m_train.yaml",
    "--preset", "proto_48m",
    "--total-steps", "100000",
    "--seq-len", "2048",
    "--data-dir", str(NVME_DATA_DIR),
    "--local-cache-dir", str(NVME_CACHE),
    "--drive-sync-dir", str(DRIVE_CKPTS),
    "--drive-logs-dir", str(DRIVE_LOGS),
    "--drive-export-dir", str(DRIVE_EXPORT),
    "--save-every-steps", "1000",
    "--save-every-minutes", "10",
    "--probe-every-steps", "500",
    "--heartbeat-seconds", "60",
    "--resume", "auto",
    "--max-consecutive-meltdowns", "3",
]

exit_code = -1
try:
    print(f"[Train] Launching: {' '.join(cmd)}", flush=True)
    proc = subprocess.run(cmd, check=False)
    exit_code = int(proc.returncode)
    if exit_code == 0:
        print(f"[Train] 100,000 steps completed cleanly! Final export: {FINAL_SAFETENSORS}", flush=True)
    elif exit_code == 42:
        print("[Train][CIRCUIT_BREAKER] Aborted after 3 consecutive unrecoverable meltdowns.", flush=True)
    else:
        raise RuntimeError(f"Training exited unexpectedly with code {exit_code}")
finally:
    if hasattr(os, "sync"):
        os.sync()
    completed_ok = exit_code == 0 and FINAL_SAFETENSORS.exists()
    melted_down = exit_code == 42
    if AUTO_UNASSIGN_RUNTIME_ON_FINISH and (completed_ok or melted_down):
        reason = "training completed & exported" if completed_ok else "3x meltdown circuit breaker"
        print(f"[Auto-Unassign] Waiting 10s after os.sync() before releasing GPU ({reason})...", flush=True)
        time.sleep(10)
        from google.colab import runtime

        runtime.unassign()
